# Task 1 - Data Cleaning and Preprocessing
**Project:** Customer Churn Analysis and Prediction &nbsp;|&nbsp; **Organisation:** Saiket Systems - Data Analysis Internship
**Task:** Task 1 of 4 &nbsp;|&nbsp; **Prepared by:** [Your Name] &nbsp;|&nbsp; **Date:** October 2026
**Dataset:** Telco Customer Churn (7,043 customers, 21 columns)

**Objective.** Prepare the raw Telco dataset for analysis and modelling: find and handle missing values, remove redundant categories, and convert categorical variables into numeric form using one-hot encoding.

## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
SEED = 42
PAL = {"No": "#4C78A8", "Yes": "#E45756"}

# Locate the project root whether this notebook sits in /notebooks or in the root folder
HERE = Path.cwd()
ROOT = next((p for p in [HERE, HERE.parent] if (p / "data" / "raw").exists()), HERE)
RAW = ROOT / "data" / "raw" / "Telco_Customer_Churn_Dataset_.csv"
PROCESSED = ROOT / "data" / "processed"; PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES = ROOT / "outputs" / "figures"; FIGURES.mkdir(parents=True, exist_ok=True)
TABLES = ROOT / "outputs" / "tables"; TABLES.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.savefig(FIGURES / name, dpi=150, bbox_inches="tight")

print("Project root:", ROOT)

Project root: /home/claude/telco-churn-project


## 1. Load and inspect the raw data

In [2]:
raw = pd.read_csv(RAW)
print("Shape:", raw.shape)
raw.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
raw.info()
print("\nColumns with null values:", int((raw.isna().sum() > 0).sum()))
print("Duplicate rows:", raw.duplicated().sum(), "| duplicate customer IDs:", raw.customerID.duplicated().sum())
raw.describe().round(2)

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

,SeniorCitizen,tenure,MonthlyCharges
count,7043.00,7043.00,7043.00
mean,0.16,32.37,64.76
std,0.37,24.56,30.09
min,0.00,0.00,18.25
25%,0.00,9.00,35.50
50%,0.00,29.00,70.35
75%,0.00,55.00,89.85
max,1.00,72.00,118.75


**Observations**
* 7,043 customers, 21 columns, no duplicate rows or IDs, and `isna()` reports no nulls.
* `TotalCharges` is stored as text (`str`) even though it should be numeric. That is a sign of hidden bad values, which `isna()` can miss.

## 2. Handle missing values

In [4]:
tc = pd.to_numeric(raw["TotalCharges"].str.strip(), errors="coerce")
blank = raw[tc.isna()]
print("Rows where TotalCharges is blank:", len(blank))
blank[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

Rows where TotalCharges is blank: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


In [5]:
print("Tenure values in these rows:", blank.tenure.unique())
print("Contract types:", blank.Contract.value_counts().to_dict())
print("Churned:", (blank.Churn == "Yes").sum())

Tenure values in these rows: [0]
Contract types: {'Two year': 10, 'One year': 1}
Churned: 0


**Decision.** All 11 blank rows have `tenure = 0`: they are new customers who have not yet received a first bill. Their cumulative charge is therefore genuinely **0**. Filling with the mean (about $2,283) would invent charges that were never billed, so imputing 0 is the logically correct choice. None of these customers churned, and they are only 0.16% of the data, so the choice has no effect on the results either way.

In [6]:
df = raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce").fillna(0.0)
print("TotalCharges dtype:", df.TotalCharges.dtype, "| missing values left:", int(df.isna().sum().sum()))

TotalCharges dtype: float64 | missing values left: 0


## 3. Remove redundant categories

In [7]:
SERVICE_COLS = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]
for c in SERVICE_COLS:
    print(f"{c:18s}", sorted(raw[c].unique()))

MultipleLines      ['No', 'No phone service', 'Yes']
OnlineSecurity     ['No', 'No internet service', 'Yes']
OnlineBackup       ['No', 'No internet service', 'Yes']
DeviceProtection   ['No', 'No internet service', 'Yes']
TechSupport        ['No', 'No internet service', 'Yes']
StreamingTV        ['No', 'No internet service', 'Yes']
StreamingMovies    ['No', 'No internet service', 'Yes']


Seven service columns carry a third value, "No internet service" (or "No phone service" for `MultipleLines`). That value repeats information already held in `InternetService` and `PhoneService`. Keeping it would create duplicate dummy columns that are perfectly correlated with `InternetService_No`. It is merged into "No". `SeniorCitizen` (0/1) is also relabelled Yes/No so every binary field looks the same.

In [8]:
for c in SERVICE_COLS:
    df[c] = df[c].replace({"No internet service": "No", "No phone service": "No"})
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

for c in SERVICE_COLS:
    assert set(df[c].unique()) == {"Yes", "No"}
df.to_csv(PROCESSED / "telco_churn_cleaned.csv", index=False)
print("Saved cleaned data ->", PROCESSED / "telco_churn_cleaned.csv")

Saved cleaned data -> /home/claude/telco-churn-project/data/processed/telco_churn_cleaned.csv


## 4. Encode categorical variables

| Column type | Technique |
|---|---|
| Target `Churn` | Yes -> 1, No -> 0 |
| Two-value columns (`gender`, `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, `SeniorCitizen`, the seven service columns) | 0 / 1 mapping |
| Multi-value columns (`InternetService`, `Contract`, `PaymentMethod`) | **One-hot encoding**, dropping the first level (`drop_first=True`) to avoid the dummy-variable trap |
| Numeric columns (`tenure`, `MonthlyCharges`, `TotalCharges`) | Kept as they are (scaling is done inside the model pipeline in Notebook 04) |

In [9]:
BINARY_COLS = ["SeniorCitizen", "Partner", "Dependents", "PhoneService", "PaperlessBilling"] + SERVICE_COLS
OHE_COLS = ["InternetService", "Contract", "PaymentMethod"]

def encode(frame):
    e = frame.drop(columns=["customerID"], errors="ignore").copy()
    if "Churn" in e:
        e["Churn"] = e["Churn"].map({"Yes": 1, "No": 0})
    e["gender"] = e["gender"].map({"Female": 0, "Male": 1})
    for c in BINARY_COLS:
        e[c] = e[c].map({"No": 0, "Yes": 1})
    return pd.get_dummies(e, columns=OHE_COLS, drop_first=True, dtype=int)

enc = encode(df)
enc.insert(0, "customerID", df["customerID"])
enc.to_csv(PROCESSED / "telco_churn_encoded.csv", index=False)
print("Encoded shape:", enc.shape)
enc.head()

Encoded shape: (7043, 25)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,0,0,1,0,1,0,0,0,1,0,0,0,0,1,29.85,29.85,0,0,0,0,0,0,1,0
1,5575-GNVDE,1,0,0,0,34,1,0,1,0,1,0,0,0,0,56.95,1889.50,0,0,0,1,0,0,0,1
2,3668-QPYBK,1,0,0,0,2,1,0,1,1,0,0,0,0,1,53.85,108.15,1,0,0,0,0,0,0,1
3,7795-CFOCW,1,0,0,0,45,0,0,1,0,1,1,0,0,0,42.30,1840.75,0,0,0,1,0,0,0,0
4,9237-HQITU,0,0,0,0,2,1,0,0,0,0,0,0,0,1,70.70,151.65,1,1,0,0,0,0,1,0


## 5. Validation

In [10]:
checks = {
    "Same number of rows as raw data": len(enc) == len(raw),
    "No missing values": enc.isna().sum().sum() == 0,
    "All feature columns numeric": enc.drop(columns="customerID").dtypes.map(lambda d: d.kind in "iuf").all(),
    "Target is only 0/1": set(enc.Churn.unique()) == {0, 1},
    "Churn count unchanged (1,869)": int(enc.Churn.sum()) == 1869,
}
for k, v in checks.items():
    print(("PASS" if v else "FAIL"), "-", k)
assert all(checks.values())

summary = pd.DataFrame({"raw": [raw.shape[1], raw.select_dtypes(exclude="number").shape[1]],
                        "encoded": [enc.shape[1] - 1, enc.drop(columns="customerID").select_dtypes(exclude="number").shape[1]]},
                       index=["columns (encoded: excl. customerID)", "text (categorical) columns"])
summary

PASS - Same number of rows as raw data
PASS - No missing values
PASS - All feature columns numeric
PASS - Target is only 0/1
PASS - Churn count unchanged (1,869)


,raw,encoded
columns (encoded: excl. customerID),21,24
text (categorical) columns,18,0


In [11]:
print("Encoded feature columns:")
print(list(enc.columns.drop(["customerID", "Churn"])))

Encoded feature columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


## Summary of Task 1
* **Missing data:** the only problem was 11 blank `TotalCharges` values hidden as text. All belong to customers with tenure 0 and were set to 0.
* **Redundancy:** "No internet service" and "No phone service" were merged into "No" in 7 columns.
* **Encoding:** the dataset went from 21 raw columns (18 text) to 23 numeric features plus the `Churn` target. The cleaned file and the encoded file are saved in `data/processed/`.
* **Validation:** row count, churn count (1,869) and numeric types were all verified, so later notebooks can rely on these files.